### Word embeddings
Word embeddings are vector representations of a particular word. \
Objective of this process is to have words with similar context occupy close spatial positions. \
For example we can use cosine as a measure of similarity: \
<br>
    <p align="center">
        <img src="cosine_similarity.png" alt="dependency tree" style="height: 484px; width:500px;"/>
    </p>
<br>
About the problems with cosine similarity: 
1. [Is Cosine-Similarity of Embeddings Really About Similarity?](https://arxiv.org/html/2403.05440v1)
2. [Problems with Cosine as a Measure of Embedding Similarity for High
Frequency Words](https://arxiv.org/pdf/2205.05092)

### Word2Vec - overview
Word2Vec is a group of related models that are used to produce word embeddings. 
1. Common Bag of Words (CBOW)
2. Skip-Gram

**CBOW** model predicts the middle word based on surrounding context words or in other words takes context of each word as the input \
and tries to predict the word corresponding to the context. \
The input(context word) is a one-hot encoded vector of size $ \textbf{V}$. \
The hiddent layer contains $ \textbf{N}$ neurons and the output is again a $ \textbf{V}$ length vector with the element being \
the softmax values. $ \textbf{W}$ is a weight matrix that maps layers together. \
The hidden layer neurons **just copy** the weighted sum of inputs to the next layer, activation function is linear, only \
non-linearity is the softmax in the output layer. \
<br>
    <p align="center">
        <img src="cbow.png" alt="dependency tree" style="height: 484px; width:500px;"/>
    </p>
<br>

For example if our corpus is only two sentences: \
Troll is great. \
Gymkata is great. \
Our network would look something like this: \
<br>
    <p align="center">
        <img src="statquest.png" alt="dependency tree" style="height: 484px; width:1000px;"/>
    </p>
<br>

Embeddings for each word are just weigths connected to each input so for Troll the vector representation is [2.02, 1.98]. \
Dimension of the vector representation is determined by the number of neurons in the hidden layer. \
For more information about word embeddings in general:\
[Word Embedding and Word2Vec, Clearly Explained!!! by StatQuest](https://www.youtube.com/watch?v=viZrOnJclY0) \
<br>
**Skip-Gram** model is inverted version of CBOW. We input word and try to predict its context words.

### Theory behind Skip-Gram model
Objective id to find word representations that are useful for predicting the surrounding words in a sentence. \
Given a sequence of training words $w_{1}, w_{2}, w_{3}, ..., w_{T}$ the objective is to maximize the average log probabilities: 
$$\dfrac{1}{T}\sum_{t=1}^{T}\sum_{\substack{-c<=j<=c, \\ j\neq{0}}} logp(w_{t+j} | w_{t})$$
Basic skip-gram defines $p(w_{t+j} | w_{t})$ using the softmax function:
$$p(w_{t+j} | w_{t}) = \dfrac{exp({v_{w_{0}}^{'}}^{T}v_{w_I})}{\sum_{w=1}^{W}{exp({v_{w}^{'}}^{T}v_{w_I})}}$$
where $v_{w}$ ans $v_{w}^{'}$ are input and output vector representations of $w$ and $W$ is the number of words in the vocabulary. \
This formulation is inpractical because the cost of computing $\nabla{logp(w_{t+j})}$ is proportional to $W$

### Alternative to softmax is hierarchical softmax
The main advantage is that instead of evaluating $W$ output nodes to obtain probability distribution it is needed to evaluate only about \
$log_{2}(W)$ \
The hierarchical softmax uses a binary tree representation of the output layer with the $W$ words as its leaves and for each nodes \
explicitly represents the relative probabilities of its child nodes. \
Each word $w$ can be reached by an appropriate path from the root of the tree. Let $n(w, j)$ be the j-th node on the path from the \
root to $w$ and let $L(w)$ be the length of this path, so $n(w, 1) = root$ and $n(w, L(w)) = w$. In addition for any inner node $n$ \
let $ch(n)$ be an arbitrary fixed child of $n$ and let $\llbracket x \rrbracket$ be 1 if $x$ is true and -1 otherwise. 
$$p(w, w_{I}) = \prod_{j=1}^{L(w)-1} \sigma(\llbracket n(w, j+1) = ch(n(w, j)) \rrbracket * {v'_{n(w, j)}}^{T}v_{w_{I}})$$
One representation $v_{w}$ for each word $w$ and one representation $v'_{n}$ for each inner node $n$ of the binary tree.
<br>
    <p align="center">
        <img src="h_softmax.png" alt="dependency tree" style="height: 484px; width:900px;"/>
    </p>
<br>
\
[Huffman tree](http://www.trevorsimonton.com/blog/2016/12/15/huffman-tree-in-word2vec.html) \
[Video explanation of hierarchical softmax](https://www.youtube.com/watch?v=pzyIWCelt_E) \
[More about hierarchical softmax and differentiated softmax](https://www.ruder.io/word-embeddings-softmax/#hierarchicalsoftmax) 

### Alternative to hierarchical softmax is negative sampling
We define negative sampling by the objective:
$$log\sigma({v'_{w_{0}}}^{T}v_{w_{I}}) + \sum_{i=1}^{k}E_{w_{i} \sim P_{n(w)}}[log\sigma(-{v'_{w_{i}}}^{T}v_{w_{I}})]$$
which replaces $logp(w_{0}|w_{I})$ in skip-gram objective. \
Thus the task is to distinguish the target word $w_{0}$ from draws from noise distribution $P_{n(w)}$ using logistic regression where there are k negative samples for \
each data sample. \
Negative sampling aims at maximizing the similarity of the words in the same context and minimizing it when they occur in different contexts. 

<br>
    <p align="center">
        <img src="negative_sampling.png" alt="dependency tree" style="height: 500px; width:900px;"/>
    </p>
<br>

So probability for positive sample is dependent only on one positive target word($\theta_{t}$) and context word($e_{c}$): 
$$p(y = 1 | context, target\;word) = \sigma(\theta_{t}^{T}e_{c})$$

[Article about negative sampling](https://aegis4048.github.io/optimize_computational_efficiency_of_skip-gram_with_negative_sampling) \
[Video by Andrew Ng about negative sampling](https://www.youtube.com/watch?v=4PXILCmVK4Q)
